In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset, get_dataset_config_names
import random

# ======================================================================
# 🧠 튜터의 비밀 코드 주석: 이 데이터셋은 무엇을 하는 곳일까요?
# ======================================================================
# 데이터셋명: jonghwanhyeon/korean-emotion-lexicon
# 의미: 한국어 감정 어휘집 (Korean Emotion Lexicon)
# 기능: 이 데이터셋은 단어(lexicon)가 어떤 감정인지, 그리고 그 단어가
#       '감정적으로 얼마나 대표적인지', '낯설지 않고 친숙한지' 등의
#       심리학적 점수(Valence, Arousal 등)를 붙여 놓은 데이터셋이에요.
# 💡 우리의 목표: 이 점수들을 분석해서, "아주 확실하고, 생생하게 느껴지는"
#       감정 단어들을 골라내는 필터링 작업을 해볼 거예요! (AI의 감정 분류 연습!)
# ======================================================================

# --- 설정값 정의 ---
DATASET_NAME = "jonghwanhyeon/korean-emotion-lexicon"
SAMPLE_COUNT = 100 # 튜터가 요청한 대로, 데이터 전체가 아닌 상위 100개만 사용합니다!

# ----------------------------------------------------------------------
# Step 1: 데이터셋 로드 및 스트리밍 테스트 (튼튼한 코드의 필수 과정!)
# ----------------------------------------------------------------------

print("💖 안녕! 오늘은 '감정'이라는 재미있는 주제로 AI 분석을 해볼 거예요.")
print("💖 데이터를 불러오기 전에, 어떤 방식으로 데이터를 가져올지 튼튼하게 확인해 봅시다.")

# 1. Config 이름 확인 (가장 먼저 할 일!)
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"\n✅ 사용 가능한 Config 목록: {configs}")
    selected_config = configs[0]
except Exception as e:
    print("\nℹ️ 데이터셋은 별도의 Config 없이 기본 설정만 사용합니다.")
    selected_config = None

# 2. 데이터 로드 (스트리밍 방식 시도)
dataset = None
try:
    # 스트리밍을 시도합니다. (성능 최적화를 위함!)
    if selected_config:
        dataset = load_dataset(DATASET_NAME, name=selected_config, split='train', streaming=True)
        print("\n✨ 성공! 스트리밍 모드로 데이터를 성공적으로 불러왔습니다. (가장 메모리 효율적이에요!)")
    else:
        dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
        print("\n✨ 성공! 스트리밍 모드로 데이터를 성공적으로 불러왔습니다.")

except Exception as e:
    # 스트리밍이 안 되면 (혹은 오류가 나면) 일반 모드로 다시 시도합니다.
    print(f"\n⚠️ 경고: 스트리밍 모드에서 오류 발생 ({type(e).__name__}). 일반 모드로 전환하여 시도합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split='train', streaming=False)
        print("✅ 성공! 일반 모드 (스트리밍 비활성화)로 데이터를 불러왔습니다.")
    except Exception as e_fallback:
        print(f"🚨 데이터 로드 실패! {e_fallback}")
        exit()

# ----------------------------------------------------------------------
# Step 2: 샘플 데이터 추출 및 준비 (전체를 다 쓸 필요는 없어요!)
# ----------------------------------------------------------------------

print(f"\n===================================================")
print(f"✨ 🎈 분석 대상: 전체 데이터 중 처음 {SAMPLE_COUNT}개의 샘플만 집중 분석해요!")
print(f"===================================================\n")

# Constraint 9, 16 적용: .take()를 사용해 상위 K개만 추출하고 리스트로 변환
sample_data_list = list(dataset.take(SAMPLE_COUNT))

# 데이터를 Pandas DataFrame으로 변환하여 분석하기 쉽도록 만듭니다.
# 데이터를 한 곳에 모아 분석하는 것이 초보자에게 가장 친숙해요!
df = pd.DataFrame(sample_data_list)

# 사용하기 편리하도록 필요한 Feature만 추출하고 이름을 바꿔줍니다.
# (튜터가 핵심 Feature만 강조해주어요!)
df_analysis = df[['lexicon', 'prototypicality', 'familiarity', 'valence', 'arousal']].copy()
df_analysis.columns = ['단어', '대표성(P)', '친숙도(F)', '가치(V)', '흥분(A)']

# ----------------------------------------------------------------------
# Step 3: 데이터 탐색 및 통계적 이해 (AI의 첫 번째 임무!)
# ----------------------------------------------------------------------

print("📊 [통계 분석]: 우리가 가져온 100개의 단어들이 가진 평균 점수를 살펴볼까요?")
print("--------------------------------------------------------------------")

# float64 컬럼에 대한 기본 통계 분석
print(f"📈 대표성(P, Prototypicality) 점수: 평균={df_analysis['대표성(P)'].mean():.3f}, 최댓값={df_analysis['대표성(P)'].max():.3f}")
print(f"😄 가치(V, Valence) 점수: 평균={df_analysis['가치(V)'].mean():.3f}, 최솟값={df_analysis['가치(V)'].min():.3f}")
print(f"⚡ 흥분(A, Arousal) 점수: 평균={df_analysis['흥분(A)'].mean():.3f}, 최댓값={df_analysis['흥분(A)'].max():.3f}")

# 흥미로운 데이터 패턴 분석: 대표성과 친숙도의 관계
print("\n🧐 [관찰] 대표성(P)과 친숙도(F)가 모두 높은 단어가 어떤가요?")
print("   -> 이 점수들이 높다는 건, 심리학적으로 '가장 전형적이고 익숙한 감정 단어'라는 뜻이에요!")

# ----------------------------------------------------------------------
# Step 4: 창의적인 실습 (조건에 맞는 '핵심 감정 단어' 찾아내기)
# ----------------------------------------------------------------------

print("\n=====================================================================")
print("✨ 💡 AI 실습 시간: '가장 강렬하고, 가장 확실한' 감정 단어 찾아내기!")
print("=====================================================================")

# 목표 정의: 
# 1. 대표성(P)이 높아야 함 (0.7 이상)
# 2. 흥분(A)가 높아야 함 (0.5 이상)
# 3. 가치(V)가 특정 범위를 만족해야 함 (0.2 ~ 0.8 사이)
# 즉, '확실하게 느껴지면서, 어느 정도 강렬한' 감정을 찾습니다!

threshold_p = 0.7
threshold_a = 0.5
min_v = 0.2
max_v = 0.8

filtered_df = df_analysis[
    (df_analysis['대표성(P)'] >= threshold_p) & 
    (df_analysis['흥분(A)'] >= threshold_a) & 
    (df_analysis['가치(V)'] >= min_v) & 
    (df_analysis['가치(V)'] <= max_v)
]

# 결과 출력
print(f"\n✅ [필터링 결과] 설정된 조건 ({threshold_p}P, {threshold_a}A, {min_v}~{max_v}V)을 만족하는 단어는 총 {len(filtered_df)}개입니다!")
print("=====================================================================")

if not filtered_df.empty:
    print("\n🔥 [가장 확실하고 생생한 감정 단어 TOP 5]:")
    # 결과가 너무 많을 경우 상위 5개만 보여주어 학습자가 지치지 않게 합니다.
    top_samples = filtered_df.head(5)
    
    print(top_samples[['단어', '대표성(P)', '친숙도(F)', '가치(V)', '흥분(A)']].to_markdown(index=False, numalign="left", stralign="left"))
    
    print("\n✨ 축하해요! 당신은 데이터의 특성을 이해하고, AI에게 필요한 '필터링 로직'을 성공적으로 구현했어요!")
else:
    print("\n💔 아쉽지만, 현재 샘플에서는 설정된 조건에 맞는 단어를 찾지 못했어요. 조건을 조금 바꿔서 다시 시도해 보세요!")

print("\n=====================================================================")
print("👏 튜터의 코멘트: 이 실습을 통해 우리는 단순히 '데이터를 불러오는 법'을 넘어,")
print("   '도메인 지식(감정 점수)'을 활용해 데이터를 분석하고 가치를 만드는 AI 분석가처럼 생각할 수 있어요!")